## Import

In [25]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm
import time
import requests

## Env

In [26]:
# load_dotenv(override=True)
# OPENROUTER_API_KEY = os.getenv["OPENROUTER_API_KEY"]

load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

## Parameters

In [46]:
RUNS = 3

MODELS = [
#    "openai/gpt-5.4-nano",
#    "openai/gpt-5.4-mini",
#    "openai/gpt-5.4",
#    "openai/gpt-5.5",
#    "openai/gpt-5.6-luna",
#    "openai/gpt-5.6-terra",
#    "openai/gpt-5.6-sol",
#    "anthropic/claude-sonnet-4.5",
#    "google/gemini-2.5-flash",
#    "google/gemma-4-31b-it",
#    "meta-llama/llama-3.3-70b-instruct", check again
    "qwen/qwen3.6-plus",
    "mistralai/mistral-large-2512",
]

# Claude:
# - anthropic/claude-sonnet-5
# - anthropic/claude-haiku-4.5

# Gemini:
# - google/gemini-3.1-pro-preview
# - google/gemini-3.5-flash
# - google/gemini-3.1-flash-lite-preview

# Gemma:
# - google/gemma-4-31b-it:free
# - google/gemma-3-27b-it

# Llama:
# - meta-llama/llama-4-maverick
# - meta-llama/llama-4-scout
# - meta-llama/llama-3.3-70b-instruct

# Qwen:
# - qwen/qwen3.7-max
# - qwen/qwen3.7-plus
# - qwen/qwen3.6-35b-a3b

# Mistral:
# - mistralai/mistral-large-2512
# - mistralai/mistral-small-2603
# - mistralai/ministral-14b-2512

TEMPERATURE = 0

SYSTEM_PROMPT = """
Person A and Person B are talking to each other. Person A asks a question and Person B answers.
Given Person A's question, estimate how strongly Person B's answer should be interpreted as
answering “yes” versus “no.”

Output a single integer score from 1 to 7, where:

1 = definitely no
2 = very likely no
3 = somewhat likely no
4 = completely uncertain, ambiguous
5 = somewhat likely yes
6 = very likely yes
7 = definitely yes

Also report the rationale for your decision in a few sentences, explaining why you gave the score you did.
If the answer is ambiguous or irrelevant, explain why.

Return score and rationale as valid JSON.
"""

SCORE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "string"},
                    "score": {
                        "type": "integer"
                    },
                    "rationale": {"type": "string"}
                },
                "required": ["id", "score", "rationale"],
                "additionalProperties": False
            }
        }
    },
    "required": ["results"],
    "additionalProperties": False
}


##  Coe

In [41]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

def read_qas(file_path: str, retain=None) -> list:
    """
    Read the data from the given CSV file and transform it into a json structure that can be used as batch input to the LLMs.

    Args:
        file_path (str): The path to the CSV file.
        retain (int): The number of rows to include in the output for direct and indirect qaüpairs respectively.
    """
    # read the csv file
    df = pd.read_csv(file_path, sep=";")

    # clip if necessary
    if retain is not None:
        if retain > len(df):
            raise ValueError(f"Retain value {retain} is greater than the number of qas in the dataframe {len(df)}.")
        df = df.head(retain)


    # extract only relevant information
    df_new = df[["CODE", "CONTEXT QUESTION", "CRITICAL UTTERANCE"]]
    df_new["text"]  = "Person A: " + df_new["CONTEXT QUESTION"] + "; Person B: " + df_new["CRITICAL UTTERANCE"]
    df_new.rename(columns={"CODE": "id"}, inplace=True)

    # convert into json
    output = [
    {
        "id": str(row["id"]),
        "text": row["text"]
    }
    for _, row in df_new.iterrows()
    ]

    return output



def score_qas(model: str, qas: list) -> dict:
    """
    Score the QAs using one OpenRouter model.
    """

    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": json.dumps(qas),
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "batch_classification_scores",
                "strict": True,
                "schema": SCORE_SCHEMA,
            },
        },
        extra_body={
            "provider": {
                "require_parameters": False,
                "allow_fallbacks": False,
            }
        },
    )

    content = response.choices[0].message.content
    result = json.loads(content)

    # ensure that the json file starts with the results field, at the top level
    if isinstance(result, list):
        result = {"results": result}
    elif not isinstance(result, dict) or "results" not in result:
        raise ValueError(f"Unexpected response shape: {result!r}")

    # validation of the score (integer between  1 and 7)
    for item in result["results"]:
        score = item["score"]
        if not isinstance(score, int) or not 1 <= score <= 7:
            raise ValueError(f"Invalid score {score!r} for item {item['id']}. Expected an integer from 1 to 7.")
        
    # fetch metadata from OpenRouter
    generation_id = response.id
    stats = get_openrouter_generation_stats(generation_id)
    metadata = {
    "generation_id": response.id,
    "requested_model": model,
    "returned_model": response.model,
    "provider_name": stats.get("provider_name"),
    "router": stats.get("router"),
    "upstream_id": stats.get("upstream_id"),
    "finish_reason": response.choices[0].finish_reason,
    "native_finish_reason": stats.get("native_finish_reason"),
    "prompt_tokens": stats.get("tokens_prompt"),
    "completion_tokens": stats.get("tokens_completion"),
    "reasoning_tokens": stats.get("native_tokens_reasoning"),
    "total_cost": stats.get("total_cost"),
    "latency": stats.get("latency"),
    "generation_time": stats.get("generation_time"),
}

    return result, metadata



def save_result_and_metadata(model: str, run: int, result, metadata):
    '''Save the scoring results to a CSV file.'''
    
    # Convert results to a dataframe
    results_df = pd.DataFrame(result["results"])
    results_df.to_csv(f"data/raw/results_{model.replace("/", "_")}_score_run{run+1}.csv", index=False)

    # Convert metedata to a dataframe
    metadata_df = pd.DataFrame([metadata])
    metadata_df.to_csv(f"data/raw/metadata_{model.replace("/", "_")}_score_run{run+1}.csv", index=False)


def get_openrouter_generation_stats(generation_id: str, max_retries: int = 5) -> dict:
    """
    Fetch OpenRouter metadata for a completed generation.
    This is where provider_name is available.
    """
    url = "https://openrouter.ai/api/v1/generation"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    }

    for attempt in range(max_retries):
        response = requests.get(
            url,
            headers=headers,
            params={"id": generation_id},
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()["data"]

        # Sometimes metadata may not be immediately available.
        if response.status_code == 404 and attempt < max_retries - 1:
            time.sleep(3)
            continue

        response.raise_for_status()

    raise RuntimeError(f"Could not fetch generation stats for {generation_id}")


In [42]:
qas = read_qas("data/external/iCO-Eval2_summarizedRatings.csv", retain=5)

print(len(qas))



5


In [ ]:
for model in MODELS:
    
    for run in tqdm(range(RUNS), desc=f"Currently collecting data from {model}"):
        result, metadata = score_qas(model, qas)
        save_result_and_metadata(model, run, result, metadata)
    
print("Data collection concluded.")
        




Currently collecting data from qwen/qwen3.6-plus: 100%|██████████| 3/3 [03:26<00:00, 68.79s/it]


Data collection concluded.


Currently collecting data from mistralai/mistral-large-2512: 100%|██████████| 3/3 [00:52<00:00, 17.59s/it]

Data collection concluded.
